# Neural Network from Scratch with NumPy

In [1]:
# import packages
import numpy as np
import pandas as pd
print("import success")

import success


In [2]:
df = pd.read_csv("customers.csv")
df.head()

,Email,Address,Avatar,Avg. Session Length,Time on App,Time on Website,Length of Membership,Yearly Amount Spent
0,mstephenson@fernandez.com,"835 Frank Tunnel\nWrightmouth, MI 82180-9605",Violet,34.497268,12.655651,39.577668,4.082621,587.951054
1,hduke@hotmail.com,"4547 Archer Common\nDiazchester, CA 06566-8576",DarkGreen,31.926272,11.109461,37.268959,2.664034,392.204933
2,pallen@yahoo.com,"24645 Valerie Unions Suite 582\nCobbborough, D...",Bisque,33.000915,11.330278,37.110597,4.104543,487.547505
3,riverarebecca@gmail.com,"1414 David Throughway\nPort Jason, OH 22070-1220",SaddleBrown,34.305557,13.717514,36.721283,3.120179,581.852344
4,mstephens@davidson-herman.com,"14023 Rodriguez Passage\nPort Jacobville, PR 3...",MediumAquaMarine,33.330673,12.795189,37.536653,4.446308,599.406092


In [3]:
new_df = df[["Time on App", "Length of Membership", "Yearly Amount Spent"]]
new_df.head()

,Time on App,Length of Membership,Yearly Amount Spent
0,12.655651,4.082621,587.951054
1,11.109461,2.664034,392.204933
2,11.330278,4.104543,487.547505
3,13.717514,3.120179,581.852344
4,12.795189,4.446308,599.406092


In [4]:
len(new_df)

500

## Initialise Layers

The sigmoid function is

$$ y = \frac{1}{1 + e^{-x}} $$

Conveniently enough, the sigmoid function also has a very simple derivative.

$$
\frac{dy}{dx} = y(1-y)
$$

Let each data point be represented by a column vector, $\begin{bmatrix}
    x \\
    y
\end{bmatrix}$. We will initialise 2 layers of 2 neurons and the last layer with 1 neuron. We can represent this with matrices.

Layer 1: $\begin{bmatrix}
    a_1 & b_1 & c_1\\
    d_1 & e_1 & f_1
\end{bmatrix}$
Layer 2: $\begin{bmatrix}
    a_2 & b_2 & c_2\\
    d_2 & e_2 & f_2
\end{bmatrix}$
Layer 3: $\begin{bmatrix}
    a_3 & b_3 & c_3
\end{bmatrix}$

Before we pass the vector into the layers, we will add a 1 at the bottom, making the vector $\begin{bmatrix}
    x \\
    y \\
    1
\end{bmatrix}$.

After it comes out of the layer, we will then apply the sigmoid function onto all of the entries in the output vector. We rinse and repeat, until it passes through the 3rd layer and the 3rd sigmoid.

To calculate the loss, we use the loss function $E = (\hat{z}- z)^2$. We can start our back propogation here.

$$ \frac{\partial E}{\partial \hat{z}} = 2(\hat{z} - z) $$

In [5]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def forward_pass(data_point, target, layer1, layer2, layer3):
    output = {}
    output["data"] = data_point
    output["layer 1 sum"] = layer1 @ np.vstack([data_point, [[1]]])
    output["layer 1 sigmoid"] = sigmoid(output["layer 1 sum"])
    output["layer 2 sum"] = layer2 @ np.vstack([output["layer 1 sigmoid"], [[1]]])
    output["layer 2 sigmoid"] = sigmoid(output["layer 2 sum"])
    output["layer 3 sum"] = (layer3 @ np.vstack([output["layer 2 sigmoid"], [[1]]])).item()
    output["layer 3 sigmoid"] = (sigmoid(output["layer 3 sum"])).item()
    output["error"] = ((output["layer 3 sigmoid"] - target) ** 2).item()
    output["error gradient"] = (2 * (output["layer 3 sigmoid"] - target)).item()
    return output

In [6]:
data_df = new_df.drop(['Yearly Amount Spent'], axis=1)
data_df.head()

,Time on App,Length of Membership
0,12.655651,4.082621
1,11.109461,2.664034
2,11.330278,4.104543
3,13.717514,3.120179
4,12.795189,4.446308


In [7]:
data = data_df.to_numpy()
targets = sigmoid(df[["Yearly Amount Spent"]].to_numpy())
print(data.shape)
print(targets.shape)

(500, 2)
(500, 1)


From the error, we need to use chain rule to find how the error changes with the weight.

$$
\begin{aligned}
\frac{\partial E}{\partial S_3} &= \frac{\partial E}{\partial \hat{Z}} \frac{\partial \hat{Z}}{\partial S_3} \\
&= 2(\hat{z} - z) O_3 (1 - O_3)
\end{aligned}
$$

Note here $\hat{Z} = O_3$. This is also a scalar.

Let's use an equation for 1 weight, $a_3$.

\begin{aligned}
\frac{\partial E}{\partial a_3} &= \frac{\partial E}{\partial S_3} \frac{\partial S_3}{\partial a_3} \\
&= 2(\hat{z} - z) O_3 (1 - O_3) O_{21}
\end{aligned}

We can expand this to the whole matrix.

$$
\begin{aligned}
\frac{\partial E}{\partial W_3} &= \frac{\partial E}{\partial S_3} \frac{\partial S_3}{\partial W_3} \\
&= 2(\hat{z} - z) O_3 (1 - O_3) \times \begin{bmatrix}
    O_{21} & O_{22} & 1
\end{bmatrix}
\end{aligned}
$$

The notation $O_{21}$ means that this is the output from the sigmoid of the 1st node in the second layer.

With the derivatives, we can update the layer 3 matrix.

$$
\text{New layer 3} = \text{layer 3} - \eta \frac{\partial E}{\partial W_3}
$$

where $\eta$ is the learning rate (essentially step size).

Now onto the second layer.

$$
\begin{aligned}
\frac{\partial E}{\partial O_2} &= \frac{\partial E}{\partial S_3} \frac{\partial S_3}{\partial O_2} \\
&=  \begin{bmatrix}
    a_3 \\
    b_3
\end{bmatrix} \frac{\partial E}{\partial S_3}\\
&= \begin{bmatrix}
    \frac{\partial E}{\partial S_3}a_3 \\
   \frac{\partial E}{\partial S_3} b_3
\end{bmatrix}
\end{aligned}
$$

$$
\begin{aligned}
\frac{\partial E}{\partial S_2} &= \frac{\partial E}{\partial O_2} \frac{\partial O_2}{\partial S_2}\\
&= \begin{bmatrix}
    \frac{\partial E}{\partial S_3}a_3 O_{21}(1 - O_{21})\\
   \frac{\partial E}{\partial S_3} b_3 O_{22}(1 - O_{22})
\end{bmatrix} \\
&= \begin{bmatrix}
    \frac{\partial E}{\partial S_{21}} \\
    \frac{\partial E}{\partial S_{22}}
\end{bmatrix}
\end{aligned}
$$

To find $\frac{\partial E}{\partial W_2}$, we need a matrix 

$$
\frac{\partial E}{\partial W_2} = 
\begin{bmatrix}
    \frac{\partial E}{\partial S_{21}} O_{11} & \frac{\partial E}{\partial S_{21}} O_{12} & \frac{\partial E}{\partial S_{21}} \\
    \frac{\partial E}{\partial S_{22}} O_{11} & \frac{\partial E}{\partial S_{22}} O_{12} & \frac{\partial E}{\partial S_{22}}
\end{bmatrix}
$$

We can achieve this using the matrix product

$$
\begin{bmatrix}
    \frac{\partial E}{\partial S_{21}} \\
    \frac{\partial E}{\partial S_{22}}
\end{bmatrix} \times \begin{bmatrix}
    O_{11} & O_{12} & 1
\end{bmatrix}
$$

$$
\text{New layer 2} = \text{layer 2} - \eta \frac{\partial E}{\partial W_2}
$$

Since the outputs of the first layer split into 2 nodes, to find the partial derivative of each output we need to do a summation.

$$
\begin{aligned}
\frac{\partial E}{\partial O_{11}} &= \frac{\partial E}{\partial S_{21}} \frac{\partial S_{21}}{\partial O_{11}} + \frac{\partial E}{\partial S_{22}} \frac{\partial S_{22}}{\partial O_{11}} \\
&= \frac{\partial E}{\partial S_{21}} a_2 + \frac{\partial E}{\partial S_{22}} d_2
\end{aligned}
$$

$$
\begin{aligned}
\frac{\partial E}{\partial O_{12}} &= \frac{\partial E}{\partial S_{21}} \frac{\partial S_{21}}{\partial O_{12}} + \frac{\partial E}{\partial S_{22}} \frac{\partial S_{22}}{\partial O_{12}} \\
&= \frac{\partial E}{\partial S_{21}} b_2 + \frac{\partial E}{\partial S_{22}} e_2
\end{aligned}
$$

To get this in matrix form, we can have

$$
\frac{\partial E}{\partial O_1} = 
\begin{bmatrix}
    a_2 & d_2 \\
    b_2 & e_2
\end{bmatrix}
\begin{bmatrix}
    \frac{\partial E}{\partial S_{21}} \\
    \frac{\partial E}{\partial S_{22}}
\end{bmatrix}
$$

We then differentiate the sigmoid.

$$
\frac{\partial E}{\partial S_1} = 
\begin{bmatrix}
    \frac{\partial E}{\partial O_{11}} O_{11} (1 - O_{11}) \\
    \frac{\partial E}{\partial O_{12}} O_{12} (1 - O_{12})
\end{bmatrix}
$$

Now we can just copy the second layer.

$$
\frac{\partial E}{\partial W_1} = 
\begin{bmatrix}
    \frac{\partial E}{\partial S_{11}} \\
    \frac{\partial E}{\partial S_{12}}
\end{bmatrix} \times \begin{bmatrix}
    x & y & 1
\end{bmatrix}
$$

$$
\text{New layer 1} = \text{layer 1} - \eta \frac{\partial E}{\partial W_1}
$$

In [8]:
def back_prop(fpd, layer1, layer2, layer3, learning_rate):
    delE_delO3 = fpd["error gradient"]
    delE_delS3 = delE_delO3 * fpd["layer 3 sigmoid"] * (1 - fpd["layer 3 sigmoid"])
    delE_delW = delE_delS3 * np.vstack([fpd["layer 2 sigmoid"], np.array([1])]).T
    new_layer3 = layer3 - (learning_rate * delE_delW)
    
    delE_delO2 = delE_delS3 * layer3[:, :-1].T
    delE_delS2 = delE_delO2 * fpd["layer 2 sigmoid"] * (np.array([1, 1]).reshape(-1, 1) - fpd["layer 2 sigmoid"])
    delE_delW = delE_delS2 @ np.vstack([fpd["layer 1 sigmoid"], np.array([1])]).T
    new_layer2 = layer2 - (learning_rate * delE_delW)

    delE_delO1 = layer2[:, :-1].T @ delE_delS2
    delE_delS1 = delE_delO1 * fpd["layer 1 sigmoid"] * (np.array([1, 1]).reshape(-1, 1) - fpd["layer 1 sigmoid"])
    delE_delW = delE_delS1 @ np.vstack([fpd["data"], np.array([1])]).T
    new_layer1 = layer1 - (learning_rate * delE_delW)

    return new_layer1, new_layer2, new_layer3

In [9]:
layer1 = np.random.rand(2, 3)
layer2 = np.random.rand(2, 3)
layer3 = np.random.rand(1, 3)

for epoch in range(250):
    counter = 1
    for i in range(len(data)):
        d = forward_pass(data[i].reshape(-1, 1), targets[i], layer1, layer2, layer3)
        layer1, layer2, layer3 = back_prop(d, layer1, layer2, layer3, 0.01)
        counter += 1
        if counter % 50 == 0 and epoch % 50 == 0:
            print(f"Error: {d["error"]:.6f}")

Error: 0.036632
Error: 0.032967
Error: 0.029898
Error: 0.027298
Error: 0.025073
Error: 0.023151
Error: 0.021478
Error: 0.020010
Error: 0.018713
Error: 0.017561
Error: 0.000420
Error: 0.000419
Error: 0.000418
Error: 0.000418
Error: 0.000417
Error: 0.000416
Error: 0.000415
Error: 0.000414
Error: 0.000413
Error: 0.000412
Error: 0.000203
Error: 0.000203
Error: 0.000203
Error: 0.000202
Error: 0.000202
Error: 0.000202
Error: 0.000202
Error: 0.000202
Error: 0.000201
Error: 0.000201
Error: 0.000133
Error: 0.000133
Error: 0.000133
Error: 0.000133
Error: 0.000133
Error: 0.000132
Error: 0.000132
Error: 0.000132
Error: 0.000132
Error: 0.000132
Error: 0.000099
Error: 0.000098
Error: 0.000098
Error: 0.000098
Error: 0.000098
Error: 0.000098
Error: 0.000098
Error: 0.000098
Error: 0.000098
Error: 0.000098
